# DGEM — Disagreement-Guided Entropy Minimization
**MIDL 2025**

Semi-supervised 3D CT segmentation, Pancreas-CT, 20% labeled.

> **Runtime → Change runtime type → T4 GPU** before running.

Set `MINI=True` (default) for a full pipeline run with synthetic data in ~10 minutes.

## 0. GPU Check

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    raise RuntimeError('No GPU — Runtime -> Change runtime type -> T4')

## 1. Install & Clone

In [ ]:
%%capture
!pip install tensorboardX nibabel h5py medpy SimpleITK scikit-image tensorboard matplotlib

In [ ]:
import os, sys
REPO = '/content/PostHocEM'
if not os.path.exists(REPO):
    !git clone https://github.com/NatalieCarlebach1/PostHocEM.git {REPO}
else:
    !git -C {REPO} pull --quiet
sys.path.insert(0, REPO)
os.chdir(REPO)
!ls

## 2. Configuration

| `MINI` | Data | Drive needed | Time |
|--------|------|-------------|------|
| `True` | Synthetic 20-case dataset (auto-generated) | No | ~10 min |
| `False` | Real Pancreas-CT from TCIA | Yes | ~12 hrs |

All paths and hyperparameters are set here.

In [ ]:
MINI = True  # True=synthetic (no Drive) | False=real Pancreas-CT (requires Drive)

SPLITS_DIR = f'{REPO}/splits/pancreas'

if MINI:
    # Everything lives in Colab's local /content — no Drive needed
    DATA_ROOT   = '/content/synthetic_h5'
    RESULT_ROOT = '/content/results'
    FIG_DIR     = '/content/figures'
    PATCH_SIZE  = 64
    MAX_EPOCHS  = 10
    EVAL_EVERY  = 2
    N_TEST      = 4
else:
    # Mount Drive for real data persistence across sessions
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT  = '/content/drive/MyDrive/DGEM'
    DATA_ROOT   = f'{DRIVE_ROOT}/pancreas_h5'
    RESULT_ROOT = f'{DRIVE_ROOT}/results'
    FIG_DIR     = f'{DRIVE_ROOT}/figures'
    PATCH_SIZE  = 96
    MAX_EPOCHS  = 300
    EVAL_EVERY  = 10
    N_TEST      = 20

BATCH_SIZE = 2

for d in [DATA_ROOT, RESULT_ROOT, SPLITS_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

BCP_SAVE  = f'{RESULT_ROOT}/bcp_baseline'
DGEM_SAVE = f'{RESULT_ROOT}/dgem_20p'
SUP_SAVE  = f'{RESULT_ROOT}/supervised_only'

mode = 'MINI — synthetic data, no Drive' if MINI else 'FULL — real Pancreas-CT + Drive'
print(f'Mode        : {mode}')
print(f'Data root   : {DATA_ROOT}')
print(f'Result root : {RESULT_ROOT}')
print(f'Patch size  : {PATCH_SIZE}  |  Max epochs: {MAX_EPOCHS}')

## 3. Data

In [ ]:
if MINI:
    print('Generating synthetic data (ellipsoid pancreas phantoms)...')
    !python data/make_synthetic.py \
        --output_dir {DATA_ROOT} \
        --n_cases    20 \
        --vol_size   {PATCH_SIZE}
else:
    # Upload raw TCIA data to Drive first, then set paths:
    RAW_DATA  = f'{DRIVE_ROOT}/raw/Pancreas-CT/PANCREAS'
    RAW_LABEL = f'{DRIVE_ROOT}/raw/TCIA_pancreas_labels'
    existing  = list(__import__('pathlib').Path(DATA_ROOT).glob('*.h5'))
    if not existing:
        !python data/preprocess_pancreas.py \
            --data_root  {RAW_DATA} \
            --label_root {RAW_LABEL} \
            --output_dir {DATA_ROOT}
    else:
        print(f'Found {len(existing)} H5 files, skipping preprocess.')

In [ ]:
# Generate train/test splits
!python data/generate_splits.py \
    --h5_dir        {DATA_ROOT} \
    --splits_dir    {SPLITS_DIR} \
    --label_percent 20 \
    --n_test        {N_TEST} \
    --seed          2020

print('Split sizes:')
for fname in ['train_lab_20.txt', 'train_unlab_20.txt', 'test.txt']:
    n = len(open(f'{SPLITS_DIR}/{fname}').readlines())
    print(f'  {fname}: {n} cases')

### Data Sanity Check

In [ ]:
import h5py, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

cases = sorted(Path(DATA_ROOT).glob('*.h5'))
assert cases, f'No H5 files in {DATA_ROOT}'

with h5py.File(str(cases[0]), 'r') as f:
    image = f['image'][:]
    label = f['label'][:]

z = int(np.argmax(label.sum(axis=(0,1))))
slices = [
    (image[:,:,z],               label[:,:,z],               'Axial'),
    (image[:,image.shape[1]//2,:], label[:,label.shape[1]//2,:], 'Coronal'),
    (image[image.shape[0]//2,:,:], label[label.shape[0]//2,:,:], 'Sagittal'),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (img_sl, lbl_sl, title) in zip(axes, slices):
    ax.imshow(img_sl, cmap='gray', vmin=0, vmax=1)
    if lbl_sl.sum() > 0:
        ax.imshow(np.ma.masked_equal(lbl_sl, 0), cmap='Reds', alpha=0.6)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle(f'Case: {cases[0].stem}  |  Shape: {image.shape}  |  Mask voxels: {label.sum()}')
plt.tight_layout()
plt.show()
print(f'Total cases: {len(cases)}')

## 4. Train BCP Baseline (SOTA — CVPR 2023)

In [ ]:
!python train_bcp_baseline.py \
    --data_root     {DATA_ROOT} \
    --splits_dir    {SPLITS_DIR} \
    --label_percent 20 \
    --max_epochs    {MAX_EPOCHS} \
    --patch_size    {PATCH_SIZE} \
    --batch_size    {BATCH_SIZE} \
    --eval_every    {EVAL_EVERY} \
    --save_dir      {BCP_SAVE} \
    --gpu           0

## 5. Train DGEM (Our Method)

In [ ]:
!python train_dgem.py \
    --data_root          {DATA_ROOT} \
    --splits_dir         {SPLITS_DIR} \
    --label_percent      20 \
    --max_epochs         {MAX_EPOCHS} \
    --patch_size         {PATCH_SIZE} \
    --batch_size         {BATCH_SIZE} \
    --em_weight          1.0 \
    --consistency_rampup 40 \
    --ema_decay          0.99 \
    --eval_every         {EVAL_EVERY} \
    --save_dir           {DGEM_SAVE} \
    --gpu                0

## 6. Ablation — Supervised Only

In [ ]:
!python train_dgem.py \
    --data_root     {DATA_ROOT} \
    --splits_dir    {SPLITS_DIR} \
    --label_percent 20 \
    --max_epochs    {MAX_EPOCHS} \
    --patch_size    {PATCH_SIZE} \
    --batch_size    {BATCH_SIZE} \
    --em_weight     0.0 \
    --eval_every    {EVAL_EVERY} \
    --save_dir      {SUP_SAVE} \
    --gpu           0

## 7. Evaluate — Paper Results Table

In [ ]:
!python evaluate.py \
    --data_root   {DATA_ROOT} \
    --test_file   {SPLITS_DIR}/test.txt \
    --num_classes 2 \
    --patch_size  {PATCH_SIZE} \
    --compare \
        "Supervised only:{SUP_SAVE}/best_model.pth" \
        "BCP (CVPR 2023):{BCP_SAVE}/best_model.pth" \
        "DGEM (ours):{DGEM_SAVE}/best_model.pth"

## 8. Loss Curves

In [ ]:
!python visualize.py losses \
    --result_dirs \
        "Supervised only:{SUP_SAVE}" \
        "BCP (CVPR 2023):{BCP_SAVE}" \
        "DGEM (ours):{DGEM_SAVE}" \
    --out_dir {FIG_DIR}

from IPython.display import Image as IPImage, display
display(IPImage(f'{FIG_DIR}/loss_curves.png'))

In [ ]:
display(IPImage(f'{FIG_DIR}/em_loss.png'))

## 9. Qualitative Results — GT vs BCP vs DGEM

For each random test case: three views (axial / coronal / sagittal) through the lesion centroid.
Each row shows the CT image with overlays: **green = GT**, **red = BCP**, **blue = DGEM**.
Dice score is shown in the subtitle of each prediction column.

In [ ]:

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, h5py, random, os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from networks import VNet
from utils.metrics import sliding_window_inference, calculate_metric_percase

# ── helpers ──────────────────────────────────────────────────────────────────
def load_model(ckpt_path, n_classes=2):
    net = VNet(n_classes=n_classes, normalization='instancenorm', has_dropout=False)
    net = nn.DataParallel(net).cuda()
    state = torch.load(ckpt_path, map_location='cuda')
    if isinstance(state, dict) and 'net' in state:
        state = state['net']
    net.load_state_dict(state)
    net.eval()
    return net

def get_centroid(label):
    coords = np.argwhere(label > 0)
    if len(coords) == 0:
        return tuple(s // 2 for s in label.shape)
    return tuple(coords.mean(0).astype(int))

def overlay(ax, img_sl, gt_sl, pred_sl, pred_color, title, dice=None):
    ax.imshow(img_sl, cmap='gray', vmin=0, vmax=1, interpolation='bilinear')
    # GT: green semi-transparent
    if gt_sl.sum() > 0:
        gt_rgba = np.zeros((*gt_sl.shape, 4))
        gt_rgba[gt_sl > 0] = [0.0, 0.9, 0.2, 0.35]
        ax.imshow(gt_rgba)
    # Prediction: coloured contour fill
    if pred_sl.sum() > 0:
        pred_rgba = np.zeros((*pred_sl.shape, 4))
        pred_rgba[pred_sl > 0] = [*pred_color, 0.55]
        ax.imshow(pred_rgba)
    lbl = title if dice is None else f'{title}\nDice={dice:.3f}'
    ax.set_title(lbl, fontsize=8, pad=3)
    ax.axis('off')

def ct_only(ax, img_sl, title):
    ax.imshow(img_sl, cmap='gray', vmin=0, vmax=1, interpolation='bilinear')
    ax.set_title(title, fontsize=8, pad=3)
    ax.axis('off')

# ── config ───────────────────────────────────────────────────────────────────
N_CASES  = 4
SEED     = 42
BCP_CLR  = [1.0, 0.2, 0.1]   # red
DGEM_CLR = [0.1, 0.4, 1.0]   # blue

random.seed(SEED)
with open(f'{SPLITS_DIR}/test.txt') as f:
    all_cases = [l.strip() for l in f if l.strip()]
cases = random.sample(all_cases, min(N_CASES, len(all_cases)))

# ── load models ──────────────────────────────────────────────────────────────
bcp_ckpt  = f'{BCP_SAVE}/best_model.pth'
dgem_ckpt = f'{DGEM_SAVE}/best_model.pth'

bcp_net  = load_model(bcp_ckpt)  if os.path.exists(bcp_ckpt)  else None
dgem_net = load_model(dgem_ckpt) if os.path.exists(dgem_ckpt) else None

if bcp_net is None:  print('WARNING: BCP checkpoint not found, skipping.')
if dgem_net is None: print('WARNING: DGEM checkpoint not found, skipping.')

# ── per-case figure ───────────────────────────────────────────────────────────
VIEWS = ['Axial', 'Coronal', 'Sagittal']

for case in cases:
    with h5py.File(str(Path(DATA_ROOT) / case), 'r') as f:
        image = f['image'][:].astype(np.float32)
        label = f['label'][:].astype(np.uint8)

    cx, cy, cz = get_centroid(label)

    bcp_pred  = sliding_window_inference(bcp_net,  image, PATCH_SIZE, 16, 8, 2)[0].astype(np.uint8) if bcp_net  else np.zeros_like(label)
    dgem_pred = sliding_window_inference(dgem_net, image, PATCH_SIZE, 16, 8, 2)[0].astype(np.uint8) if dgem_net else np.zeros_like(label)

    bcp_dice  = calculate_metric_percase(bcp_pred,  label)[0]
    dgem_dice = calculate_metric_percase(dgem_pred, label)[0]

    # 3 views x 4 cols: [CT only | GT overlay | BCP | DGEM]
    fig, axes = plt.subplots(3, 4, figsize=(14, 9))
    fig.patch.set_facecolor('#111111')
    fig.suptitle(f'Case: {Path(case).stem}   |   BCP Dice={bcp_dice:.3f}   DGEM Dice={dgem_dice:.3f}',
                 fontsize=11, color='white', y=0.98)

    for row, view in enumerate(VIEWS):
        if view == 'Axial':
            img_sl = image[:, :, cz];  gt_sl = label[:, :, cz]
            bp_sl  = bcp_pred[:, :, cz]; dp_sl = dgem_pred[:, :, cz]
        elif view == 'Coronal':
            img_sl = image[:, cy, :];  gt_sl = label[:, cy, :]
            bp_sl  = bcp_pred[:, cy, :]; dp_sl = dgem_pred[:, cy, :]
        else:
            img_sl = image[cx, :, :];  gt_sl = label[cx, :, :]
            bp_sl  = bcp_pred[cx, :, :]; dp_sl = dgem_pred[cx, :, :]

        ct_only(axes[row, 0], img_sl, f'{view} — CT')
        overlay(axes[row, 1], img_sl, gt_sl, gt_sl,  [0.0, 0.9, 0.2], f'{view} — Ground Truth')
        overlay(axes[row, 2], img_sl, gt_sl, bp_sl,  BCP_CLR,         f'{view} — BCP (SOTA)',  bcp_dice)
        overlay(axes[row, 3], img_sl, gt_sl, dp_sl,  DGEM_CLR,        f'{view} — DGEM (ours)', dgem_dice)

        for ax in axes[row]:
            ax.set_facecolor('#111111')
            for spine in ax.spines.values():
                spine.set_edgecolor('#333333')

    # Legend
    legend_els = [
        mpatches.Patch(color=[0.0, 0.9, 0.2], alpha=0.5, label='Ground Truth'),
        mpatches.Patch(color=BCP_CLR,          alpha=0.6, label='BCP (SOTA)'),
        mpatches.Patch(color=DGEM_CLR,         alpha=0.6, label='DGEM (ours)'),
    ]
    fig.legend(handles=legend_els, loc='lower center', ncol=3,
               fontsize=9, framealpha=0.2, labelcolor='white',
               facecolor='#222222', bbox_to_anchor=(0.5, 0.01))

    plt.subplots_adjust(wspace=0.04, hspace=0.18, bottom=0.07)
    out_path = f'{FIG_DIR}/qual_{Path(case).stem}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved: {out_path}')


## 10. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RESULT_ROOT}

## 11. Smoke Test (No Data, <60s)

Full forward/backward pass with random tensors. Run this first to confirm the code is correct.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from networks import VNet
from utils.losses import SupLoss, entropy_loss_masked
from utils.ramps  import get_current_consistency_weight
from utils.metrics import sliding_window_inference, calculate_metric_percase

torch.manual_seed(42)
P, B, dev = 64, 2, 'cuda'

# Build models
net     = VNet(n_classes=2, normalization='instancenorm', has_dropout=True).to(dev)
net_ema = VNet(n_classes=2, normalization='instancenorm', has_dropout=True).to(dev)
net_ema.load_state_dict(net.state_dict())
for param in net_ema.parameters(): param.detach_()

optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
criterion = SupLoss(n_classes=2)

lab_img   = torch.randn(B,1,P,P,P).to(dev)
lab_lbl   = torch.randint(0,2,(B,P,P,P)).to(dev)
unlab_img = torch.randn(B,1,P,P,P).to(dev)

net.train()
net_ema.eval()

# 1. Supervised loss
sup_loss = criterion(net(lab_img)[0], lab_lbl)

# 2. DGEM: entropy on disagreement voxels
student_probs = F.softmax(net(unlab_img)[0], dim=1)
student_pred  = student_probs.argmax(dim=1)
with torch.no_grad():
    teacher_pred = F.softmax(net_ema(unlab_img)[0], dim=1).argmax(dim=1)
disagree = (student_pred != teacher_pred).float()
em_loss  = entropy_loss_masked(student_probs, disagree)
lam      = get_current_consistency_weight(1, 1.0, 40)
total    = sup_loss + lam * em_loss
optimizer.zero_grad()
total.backward()
optimizer.step()

# 3. EMA update
with torch.no_grad():
    for sp, tp in zip(net.parameters(), net_ema.parameters()):
        tp.data = 0.99 * tp.data + 0.01 * sp.data

# 4. Sliding window inference
net.eval()
vol  = np.random.rand(P,P,P).astype(np.float32)
pred, score = sliding_window_inference(net, vol, P, 16, 8, 2)
assert pred.shape == (P,P,P), 'Pred shape wrong'
assert score.shape == (2,P,P,P), 'Score shape wrong'

# 5. Metrics
gt = (np.random.rand(P,P,P) > 0.8).astype(np.uint8)
pr = (np.random.rand(P,P,P) > 0.8).astype(np.uint8)
d, jc, hd, asd = calculate_metric_percase(pr, gt)

print('=' * 45)
print('ALL CHECKS PASSED')
print(f'  sup_loss       : {sup_loss.item():.4f}')
print(f'  em_loss        : {em_loss.item():.4f}')
print(f'  disagree_ratio : {disagree.mean().item():.3f}')
print(f'  lam            : {lam:.4f}')
print(f'  total_loss     : {total.item():.4f}')
print(f'  pred shape     : {pred.shape}')
print(f'  score shape    : {score.shape}')
print(f'  Dice (random)  : {d:.4f}')
print('=' * 45)

## 12. Download Results

- **MINI mode**: downloads a zip of all figures directly to your browser.
- **FULL mode**: figures are already saved on Drive. The zip is also downloaded.

In [ ]:
from google.colab import files
import zipfile
zip_path = '/content/figures.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in __import__('pathlib').Path(FIG_DIR).glob('*.png'):
        zf.write(str(f), f.name)
files.download(zip_path)
print('Download started.')

# Also download best checkpoints in MINI mode
if MINI:
    ckpt_zip = '/content/checkpoints.zip'
    with zipfile.ZipFile(ckpt_zip, 'w') as zf:
        for p in __import__('pathlib').Path(RESULT_ROOT).rglob('best_model.pth'):
            zf.write(str(p), str(p.relative_to(RESULT_ROOT)))
    files.download(ckpt_zip)
    print('Checkpoints download started.')